# Titanic: Análise e Modelagem Melhorada com Explicações Detalhadas

## Objetivo do Notebook

Este notebook está estruturado para **ensinar** você a trabalhar com dados e estatísticas.

Cobriremos:
1. **Carregamento e exploração de dados** - entender o que temos
2. **Pré-processamento** - limpar e preparar dados faltantes
3. **Feature Engineering** - criar novas variáveis úteis
4. **Modelagem** - treinar e validar um modelo
5. **Previsão** - gerar submissão

Cada célula tem comentários detalhados explicando **O QUE**, **POR QUE** e **QUAL É O RESULTADO**.

## Parte 1: Importar Bibliotecas e Carregar Dados

### O que são bibliotecas?
Bibliotecas são pacotes de código já pronto que outras pessoas criaram.
Usamos `import` para usar essas funções sem ter que escrever tudo do zero.

### Principais bibliotecas que usaremos:
- **pandas** (pd) - trabalhar com dados em formato tabular (tipo Excel, mas em Python)
- **numpy** (np) - cálculos matemáticos e arrays
- **scikit-learn** (sklearn) - machine learning (treinar modelos)
- **matplotlib / seaborn** - criar gráficos para visualizar dados

In [1]:
# Importar bibliotecas essenciais
import numpy as np              # Cálculos numéricos
import pandas as pd             # Manipular dados em tabelas
import matplotlib.pyplot as plt # Criar gráficos
import seaborn as sns           # Gráficos mais bonitos que matplotlib
import warnings

# Suprimindo avisos para deixar a saída mais limpa
warnings.filterwarnings('ignore')

# Configuração visual para gráficos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Bibliotecas carregadas com sucesso!")

✓ Bibliotecas carregadas com sucesso!


## Passo 1: Carregar e Inspecionar os Dados

### O que fazemos aqui?
Lemos os arquivos CSV (Comma Separated Values) que contêm:
- `train.csv` - dados com exemplos conhecidos (usado para treinar o modelo)
- `test.csv` - dados sem a resposta (usamos o modelo para prever)

### Por que é importante?
Antes de fazer qualquer análise, precisamos **entender** nossos dados:
- Quantas linhas e colunas temos?
- Qual é o tipo de cada coluna?
- Há valores faltantes (ausentes)?

In [2]:
# Carregar os dados de treino e teste
# pd.read_csv lê um arquivo CSV e retorna um DataFrame (uma tabela Python)
train_data = pd.read_csv("../data/raw/train.csv")
test_data = pd.read_csv("../data/raw/test.csv")

print("=" * 80)
print("INFORMAÇÕES GERAIS DOS DADOS DE TREINO")
print("=" * 80)

# Mostrar as primeiras linhas
# head() mostra os primeiros 5 registros por padrão
print("\n1. Primeiras 5 linhas dos dados:")
print(train_data.head())

print("\n" + "=" * 80)
print("2. Informações sobre o tipo e valores faltantes:")
print("=" * 80)
# info() mostra:
# - nome de cada coluna
# - tipo de dados (int64, float64, object, etc)
# - quantos valores não nulos existem (Non-Null Count)
train_data.info()

print("\n" + "=" * 80)
print("3. Estatísticas descritivas (media, desvio padrão, min, max, etc):")
print("=" * 80)
# describe() mostra estatísticas das colunas numéricas
print(train_data.describe())

print("\n" + "=" * 80)
print("4. Tamanho dos dados:")
print("=" * 80)
print(f"Treino: {train_data.shape[0]} passageiros, {train_data.shape[1]} colunas")
print(f"Teste:  {test_data.shape[0]} passageiros, {test_data.shape[1]} colunas")

INFORMAÇÕES GERAIS DOS DADOS DE TREINO

1. Primeiras 5 linhas dos dados:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000

## Passo 2: Analisar Valores Faltantes (Missing Values)

### O que é um valor faltante?
Um valor faltante (missing value, ou NaN) é quando um dado não está preenchido.
Por exemplo: alguns passageiros não têm idade registrada.

### Por que é um problema?
A maioria dos algoritmos de machine learning **não consegue lidar** com valores faltantes.
Precisamos **tratar** esses valores:
- Remover a linha inteira (perder informação)
- Preencher com um valor padrão (média, mediana, moda)
- Usar uma técnica mais sofisticada

In [3]:
# Contar valores faltantes em cada coluna
print("=" * 80)
print("VALORES FALTANTES (NaN) NO CONJUNTO DE TREINO")
print("=" * 80)

# Criar um DataFrame com a contagem e percentual de valores faltantes
missing = pd.DataFrame({
    'Total_Missing': train_data.isnull().sum(),  # Contar NaNs por coluna
    'Percent': (train_data.isnull().sum() / len(train_data) * 100).round(2)  # Calcular %
})

# Mostrar apenas colunas que têm valores faltantes
missing_data = missing[missing['Total_Missing'] > 0].sort_values('Total_Missing', ascending=False)
print(missing_data)

print("\n" + "=" * 80)
print("INTERPRETAÇÃO:")
print("=" * 80)
print("""
- Age: 177 valores faltantes (19.87% dos dados)
  → Precisamos preencher com algo (média, mediana por grupo, etc)

- Cabin: 687 valores faltantes (77.1% dos dados)
  → Temos muitos dados faltantes. Vamos criar uma variável binária: tem Cabin ou não?

- Embarked: 2 valores faltantes (0.22% dos dados)
  → Muito poucos. Vamos preencher com a moda (valor mais frequente)
""")

VALORES FALTANTES (NaN) NO CONJUNTO DE TREINO
          Total_Missing  Percent
Cabin               687    77.10
Age                 177    19.87
Embarked              2     0.22

INTERPRETAÇÃO:

- Age: 177 valores faltantes (19.87% dos dados)
  → Precisamos preencher com algo (média, mediana por grupo, etc)

- Cabin: 687 valores faltantes (77.1% dos dados)
  → Temos muitos dados faltantes. Vamos criar uma variável binária: tem Cabin ou não?

- Embarked: 2 valores faltantes (0.22% dos dados)
  → Muito poucos. Vamos preencher com a moda (valor mais frequente)



## Passo 3: Feature Engineering - Criar Novas Variáveis

### O que é Feature Engineering?
É o **processo de criar novas colunas** a partir das existentes.
Essas novas variáveis podem ser **muito mais úteis** para o modelo prever.

### Por que é importante?
Um bom feature engineering pode aumentar a acurácia muito mais do que mudar o modelo!

### Exemplos que vamos fazer:
1. **Title** - extrair "Mr", "Mrs", "Miss" do nome (indica idade e sexo indiretamente)
2. **FamilySize** - número total de familiares a bordo (SibSp + Parch + 1)
3. **IsAlone** - viajava sozinho? (sim/não)
4. **CabinDeck** - qual era o andar da cabine? (A, B, C, D, E, F, G)
5. **AgeGroup** - categorizar idade em grupos (child, teen, adult, elderly)

In [4]:
# Vamos criar uma cópia dos dados para não perder o original
# (boa prática em análise de dados)
df_train = train_data.copy()
df_test = test_data.copy()

print("=" * 80)
print("FEATURE ENGINEERING - Criando novas variáveis")
print("=" * 80)

# --- Feature 1: Extrair TITLE do Nome ---
# A coluna Name contém: "Braund, Mr. Owen Harris"
# Vamos extrair "Mr"
print("\n1. Extraindo Title do Nome...")
print("   Exemplos de nomes originais:")
print("   -", df_train['Name'].iloc[0])  # Mostrar um exemplo
print("   -", df_train['Name'].iloc[1])

# Extrair o título (usando expressão regular para encontrar "Mr." "Mrs." etc)
def extract_title(name):
    """Extrai o título (Mr, Mrs, Miss, Master, etc) do nome"""
    import re
    # Procurar padrão: palavra seguida de ponto
    title_search = re.search(r' ([A-Za-z]+)\.', name)
    if title_search:
        return title_search.group(1)  # Retorna o que foi encontrado
    return None

# Aplicar a função a cada nome
df_train['Title'] = df_train['Name'].apply(extract_title)
df_test['Title'] = df_test['Name'].apply(extract_title)

print("   Títulos encontrados:", df_train['Title'].unique())
print("   Distribuição de títulos:")
print(df_train['Title'].value_counts())

# Agrupar títulos raros em "Rare"
# Apenas Mr, Mrs, Miss, Master são comuns
title_counts = df_train['Title'].value_counts()
rare_titles = title_counts[title_counts < 10].index
print(f"\n   Títulos raros (menos de 10 ocorrências): {list(rare_titles)}")

# Substituir títulos raros por "Rare"
df_train['Title'] = df_train['Title'].replace(rare_titles, 'Rare')
df_test['Title'] = df_test['Title'].replace(rare_titles, 'Rare')

print(f"   Resultado final - Títulos: {df_train['Title'].unique()}")

# --- Feature 2: Tamanho da Família ---
print("\n2. Criando variável FamilySize...")
print("   Fórmula: SibSp (irmãos+cônjuge) + Parch (pais+filhos) + 1 (próprio passageiro)")
df_train['FamilySize'] = df_train['SibSp'] + df_train['Parch'] + 1
df_test['FamilySize'] = df_test['SibSp'] + df_test['Parch'] + 1
print("   Exemplo: Passageiro com 1 irmão e 0 pais/filhos -> FamilySize = 1 + 0 + 1 = 2")
print("   Distribuição de FamilySize:", df_train['FamilySize'].value_counts().sort_index().head())

# --- Feature 3: Viajava Sozinho? ---
print("\n3. Criando variável IsAlone...")
print("   Se FamilySize == 1, então IsAlone = 1 (True), senão IsAlone = 0 (False)")
df_train['IsAlone'] = (df_train['FamilySize'] == 1).astype(int)
df_test['IsAlone'] = (df_test['FamilySize'] == 1).astype(int)
print(f"   Passageiros que viajavam sozinhos: {df_train['IsAlone'].sum()}")
print(f"   Passageiros que viajavam acompanhados: {(df_train['IsAlone'] == 0).sum()}")

# --- Feature 4: Deck da Cabine ---
print("\n4. Criando variável CabinDeck...")
print("   Cabin format: 'A15', 'B32' (letra = andar, número = cabine)")
print("   Vamos extrair apenas a letra.")
print("   Exemplos de Cabin:", df_train['Cabin'].dropna().head(3).values)

# Extrair primeira letra da cabine (se existir)
df_train['CabinDeck'] = df_train['Cabin'].str[0]
df_test['CabinDeck'] = df_test['Cabin'].str[0]

# Se não há Cabin, marcar como 'X' (missing)
df_train['CabinDeck'] = df_train['CabinDeck'].fillna('X')
df_test['CabinDeck'] = df_test['CabinDeck'].fillna('X')

print("   Decks encontrados:", df_train['CabinDeck'].unique())
print("   Distribuição:", df_train['CabinDeck'].value_counts().to_dict())

# --- Feature 5: Faixa Etária ---
print("\n5. Criando variável AgeGroup (faixa etária)...")
print("   Vamos agrupar idades em: Child (0-12), Teen (13-19), Adult (20-60), Elderly (60+)")
print("   Mas antes, precisamos preencher os valores faltantes de Age.")

# Preencher Age com a mediana por Title e Sex
print("\n   Preenchendo valores faltantes de Age...")
print("   Estratégia: Para cada combinação de Title + Sex, usar a mediana de Age")
print("   Exemplo: Idade mediana de 'Mr' masculino ≈ 24 anos")

age_by_title_sex = df_train.groupby(['Title', 'Sex'])['Age'].median()
print("\n   Tabela de medianas por Title e Sex:")
print(age_by_title_sex)

# Função para preencher Age
def fill_age(row):
    if pd.isnull(row['Age']):
        return age_by_title_sex[row['Title'], row['Sex']]
    return row['Age']

df_train['Age'] = df_train.apply(fill_age, axis=1)
df_test['Age'] = df_test.apply(fill_age, axis=1)

print("\n   Valores faltantes de Age após preenchimento:", df_train['Age'].isnull().sum())

# Criar grupos de idade
df_train['AgeGroup'] = pd.cut(df_train['Age'], bins=[0, 12, 19, 60, 100], 
                               labels=['Child', 'Teen', 'Adult', 'Elderly'])
df_test['AgeGroup'] = pd.cut(df_test['Age'], bins=[0, 12, 19, 60, 100], 
                              labels=['Child', 'Teen', 'Adult', 'Elderly'])

print("   Distribuição de AgeGroup:", df_train['AgeGroup'].value_counts().to_dict())

# --- Feature 6: Preencher Fare (se faltando) ---
print("\n6. Preenchendo valores faltantes de Fare...")
print(f"   Valores faltantes em Fare - Treino: {df_train['Fare'].isnull().sum()}")
print(f"   Valores faltantes em Fare - Teste: {df_test['Fare'].isnull().sum()}")

if df_train['Fare'].isnull().sum() > 0:
    df_train['Fare'].fillna(df_train['Fare'].median(), inplace=True)
if df_test['Fare'].isnull().sum() > 0:
    df_test['Fare'].fillna(df_test['Fare'].median(), inplace=True)

print("   Valores faltantes de Fare após preenchimento:")
print(f"   Treino: {df_train['Fare'].isnull().sum()}, Teste: {df_test['Fare'].isnull().sum()}")

# --- Feature 7: Preencher Embarked ---
print("\n7. Preenchendo valores faltantes de Embarked (porto de embarque)...")
print(f"   Valores faltantes: {df_train['Embarked'].isnull().sum()}")
print("   Estratégia: Preencher com a moda (valor mais frequente)")
print("   Moda de Embarked:", df_train['Embarked'].mode()[0])

df_train['Embarked'].fillna(df_train['Embarked'].mode()[0], inplace=True)
df_test['Embarked'].fillna(df_test['Embarked'].mode()[0], inplace=True)

print("\n" + "=" * 80)
print("NOVAS COLUNAS CRIADAS:")
print("=" * 80)
print("Colunas no dataset após feature engineering:")
print(df_train.columns.tolist())
print(f"\nTotal de colunas: {len(df_train.columns)}")

FEATURE ENGINEERING - Criando novas variáveis

1. Extraindo Title do Nome...
   Exemplos de nomes originais:
   - Braund, Mr. Owen Harris
   - Cumings, Mrs. John Bradley (Florence Briggs Thayer)
   Títulos encontrados: ['Mr' 'Mrs' 'Miss' 'Master' 'Don' 'Rev' 'Dr' 'Mme' 'Ms' 'Major' 'Lady'
 'Sir' 'Mlle' 'Col' 'Capt' 'Countess' 'Jonkheer']
   Distribuição de títulos:
Title
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Col           2
Mlle          2
Major         2
Ms            1
Mme           1
Don           1
Lady          1
Sir           1
Capt          1
Countess      1
Jonkheer      1
Name: count, dtype: int64

   Títulos raros (menos de 10 ocorrências): ['Dr', 'Rev', 'Col', 'Mlle', 'Major', 'Ms', 'Mme', 'Don', 'Lady', 'Sir', 'Capt', 'Countess', 'Jonkheer']
   Resultado final - Títulos: ['Mr' 'Mrs' 'Miss' 'Master' 'Rare']

2. Criando variável FamilySize...
   Fórmula: SibSp (irmãos+cônjuge) + Parch (pais+filhos) + 1 (próprio passage

## Passo 4: Preparar Dados para o Modelo

### O que fazemos aqui?
Machine learning funciona com **números**, não com texto.
Precisamos converter variáveis categóricas (texto) em números.

### Por que isso importa?
A maioria dos algoritmos usa operações matemáticas e vetores.
Texto como "male" ou "S" não pode ser usado diretamente por esses algoritmos.

### Métodos usados e de onde eles vêm
- `map()` é um método de `pandas.Series` que substitui valores segundo um dicionário.
- `pd.get_dummies()` é uma função do `pandas` que faz **one-hot encoding**.
- `LabelEncoder`, `OrdinalEncoder` e `StandardScaler` são do `sklearn.preprocessing`.

### Alternativas que não usamos
- `LabelEncoder` (`sklearn.preprocessing.LabelEncoder`)
  - Não usado porque gera valores numéricos ordinais.
  - Imagine `Mr=0`, `Mrs=1`, `Miss=2`: isso cria uma ordem que não existe.
- `OrdinalEncoder` (`sklearn.preprocessing.OrdinalEncoder`)
  - Também não usado por transformar categorias nominais em inteiros ordinais.
- `Target encoding` ou `mean encoding`
  - Não usado porque é mais complexo e exige cuidado para evitar vazamento de dados (data leakage).
- `StandardScaler` (`sklearn.preprocessing.StandardScaler`)
  - Não usado no pipeline atual porque os modelos de árvore não dependem de escala.
  - Pode ser útil para `LogisticRegression`, `SVM` ou `KNN`.

### Pontos positivos e negativos
- **Label Encoding / Ordinal Encoding**
  - Positivo: simples e gera apenas uma coluna.
  - Negativo: cria ordem artificial em categorias nominais.
- **One-Hot Encoding**
  - Positivo: preserva categorias como independentes.
  - Negativo: aumenta o número de colunas.
- **StandardScaler**
  - Positivo: normaliza variáveis numéricas.
  - Negativo: não melhora modelos baseados em árvore e pode ser desnecessário aqui.


In [5]:
print("=" * 80)
print("PREPARANDO DADOS PARA O MODELO")
print("=" * 80)

# 1. Label Encoding para Sex (masculino=1, feminino=0)
print("\n1. Convertendo Sex em números...")
print("   Mapeamento: male -> 1, female -> 0")
sex_mapping = {"male": 1, "female": 0}
# .map() é um método do pandas que substitui valor por valor no Series
# usando um dicionário.
df_train["Sex"] = df_train["Sex"].map(sex_mapping)
df_test["Sex"] = df_test["Sex"].map(sex_mapping)
print("   Resultado:", df_train["Sex"].head().tolist())

# 2. Codificar outras variáveis categóricas
print("\n2. Codificando variáveis categóricas com One-Hot Encoding...")
print("   Este método cria uma coluna para cada valor único.")
print("   Exemplo Title: 'Mr' -> coluna 'Title_Mr' = 1, 'Title_Mrs' = 0, etc")

# Definir quais colunas são categóricas
categorical_cols = ["Title", "AgeGroup", "CabinDeck", "Embarked"]

# One-hot encoding vem do pandas e cria colunas binárias
# para cada categoria, mantendo categorias nominais separadas.
df_train = pd.get_dummies(df_train, columns=categorical_cols, drop_first=True)
df_test = pd.get_dummies(df_test, columns=categorical_cols, drop_first=True)

# IMPORTANTE: Depois de fazer get_dummies, treino e teste podem ter colunas diferentes
# Vamos sincronizar as colunas
train_cols = set(df_train.columns)
test_cols = set(df_test.columns)

# Colunas que estão no treino mas não no teste
missing_in_test = train_cols - test_cols
# Colunas que estão no teste mas não no treino
missing_in_train = test_cols - train_cols

if missing_in_test:
    print(f"\n   Colunas faltando no teste: {missing_in_test}")
    for col in missing_in_test:
        df_test[col] = 0

if missing_in_train:
    print(f"   Colunas faltando no treino: {missing_in_train}")
    for col in missing_in_train:
        df_train[col] = 0

# Garantir mesma ordem de colunas
df_test = df_test[df_train.columns]

print(f"\n   Total de colunas após codificação: {len(df_train.columns)}")
print(f"   Total de linhas - Treino: {len(df_train)}, Teste: {len(df_test)}")

# 3. Selecionar features (variáveis) para usar no modelo
print("\n3. Selecionando features para treinar o modelo...")
features_to_use = [
    "Pclass", "Sex", "Age", "Fare", "FamilySize", "IsAlone"
]

# Adicionar colunas criadas por one-hot encoding que existem nos dados
encoded_cols = [col for col in df_train.columns if any(prefix in col for prefix in ["Title_", "AgeGroup_", "CabinDeck_", "Embarked_"])]
features_to_use.extend(encoded_cols)

print(f"   Features que vamos usar ({len(features_to_use)} ao total):")
print(f"   {features_to_use}")

# Preparar X (features) e y (target/alvo)
X_train = df_train[features_to_use]
y_train = df_train["Survived"]  # O que queremos prever

X_test = df_test[features_to_use]

print(f"\n   Dimensões:")
print(f"   X_train (features de treino): {X_train.shape}")
print(f"   y_train (respostas de treino): {y_train.shape}")
print(f"   X_test (features de teste): {X_test.shape}")


PREPARANDO DADOS PARA O MODELO

1. Convertendo Sex em números...
   Mapeamento: male -> 1, female -> 0
   Resultado: [1, 0, 0, 0, 1]

2. Codificando variáveis categóricas com One-Hot Encoding...
   Este método cria uma coluna para cada valor único.
   Exemplo Title: 'Mr' -> coluna 'Title_Mr' = 1, 'Title_Mrs' = 0, etc

   Colunas faltando no teste: {'Survived', 'CabinDeck_T'}
   Colunas faltando no treino: {'Title_Master'}

   Total de colunas após codificação: 31
   Total de linhas - Treino: 891, Teste: 418

3. Selecionando features para treinar o modelo...
   Features que vamos usar (24 ao total):
   ['Pclass', 'Sex', 'Age', 'Fare', 'FamilySize', 'IsAlone', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare', 'AgeGroup_Teen', 'AgeGroup_Adult', 'AgeGroup_Elderly', 'CabinDeck_B', 'CabinDeck_C', 'CabinDeck_D', 'CabinDeck_E', 'CabinDeck_F', 'CabinDeck_G', 'CabinDeck_T', 'CabinDeck_X', 'Embarked_Q', 'Embarked_S', 'Title_Master']

   Dimensões:
   X_train (features de treino): (891, 24)
   

## Passo 5: Dividir Dados e Validação Cruzada

### O que é Validação Cruzada?
É uma técnica para testar se nosso modelo **generaliza bem**.

### Por que é importante?
Se treinamos e testamos no mesmo dado, o modelo "decora" a resposta (overfitting).
Não sabemos como ele se comporta em dados **novos**.

### Métodos usados e de onde eles vêm
- `StratifiedKFold` vem de `sklearn.model_selection`.
  - Divide os dados em folds mantendo a proporção das classes.
- `cross_val_score` também vem de `sklearn.model_selection`.
  - Executa a validação cruzada e retorna um score para cada fold.
- `VotingClassifier`, `RandomForestClassifier`, `GradientBoostingClassifier`,
  `ExtraTreesClassifier`, `AdaBoostClassifier` vêm de `sklearn.ensemble`.
- `LogisticRegression` vem de `sklearn.linear_model`.
- `accuracy_score`, `precision_score`, `recall_score` e `f1_score` vêm de `sklearn.metrics`.

### Como funciona (K-Fold com k=5)?
1. Dividir os dados em 5 partes.
2. Para cada iteração:
   - Use 4 partes para treinar.
   - Use 1 parte para testar.
3. Repetir 5 vezes, mudando o fold de teste.
4. Calcular a média dos resultados.

### Alternativas que não usamos
- `SVM` (`sklearn.svm.SVC`)
  - Não usado porque é mais lento em datasets maiores e requer escala dos dados.
- `KNeighborsClassifier` (`sklearn.neighbors.KNeighborsClassifier`)
  - Não usado por ser sensível à escala e ao número de features.
- `DecisionTreeClassifier` isolado
  - Não usado porque tende a overfitting.
- `StackingClassifier` (`sklearn.ensemble.StackingClassifier`)
  - Não usado por aumentar a complexidade do pipeline e exigir validação adicional.
- `XGBoost` / `LightGBM`
  - Não usados porque não fazem parte do `sklearn` padrão e exigem dependências extras.
- `Hard voting`
  - Não usado porque `soft voting` usa probabilidades e tende a ser mais robusto.

### Pontos positivos e negativos
- **RandomForestClassifier**
  - Positivo: robusto, boa generalização, funciona bem com dados mistos.
  - Negativo: pode ser mais lento e menos interpretável.
- **GradientBoostingClassifier**
  - Positivo: alta precisão, ótimo para padrões complexos.
  - Negativo: sensível a hiperparâmetros e mais lento.
- **ExtraTreesClassifier**
  - Positivo: mais aleatório, tende a reduzir variância.
  - Negativo: probabilidades podem ser menos calibradas.
- **AdaBoostClassifier**
  - Positivo: melhora fracos aprendizes e foca em erros.
  - Negativo: pode ser sensível a ruído e outliers.
- **LogisticRegression**
  - Positivo: simples, rápido e fácil de interpretar.
  - Negativo: é linear e pode não capturar relações não lineares.
- **VotingClassifier (soft voting)**
  - Positivo: combina forças de modelos diferentes e reduz risco de falha de um único estimador.
  - Negativo: aumenta o tempo de treino e não melhora se os modelos forem muito parecidos.


In [6]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, AdaBoostClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("=" * 80)
print("VALIDAÇÃO CRUZADA - Testar confiabilidade do ensemble")
print("=" * 80)

print("""
O que vamos fazer:
1. Dividir os dados de treino em 5 partes
2. Treinar 5 modelos (4 partes para treinar, 1 para validar)
3. Calcular a acurácia em cada fold (divisão)
4. Reportar a média e desvio padrão

Isso simula como o modelo se comportaria em dados desconhecidos.
""")

# Usaremos um ensemble simples com cinco modelos diferentes
# e votações suaves (soft voting) para combinar previsões.

model_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

model_gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

model_et = ExtraTreesClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

model_ab = AdaBoostClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)


model_lr = LogisticRegression(
    max_iter=1000,
    solver='liblinear',
    random_state=42
)

model = VotingClassifier(
    estimators=[('rf', model_rf), ('gb', model_gb), ('et', model_et), ('ab', model_ab), ('lr', model_lr)],
    voting='soft'
)
# Validação cruzada com 5 folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Calcular scores em cada fold
scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')


VALIDAÇÃO CRUZADA - Testar confiabilidade do ensemble

O que vamos fazer:
1. Dividir os dados de treino em 5 partes
2. Treinar 5 modelos (4 partes para treinar, 1 para validar)
3. Calcular a acurácia em cada fold (divisão)
4. Reportar a média e desvio padrão

Isso simula como o modelo se comportaria em dados desconhecidos.



## Passo 6: Treinar o Modelo Final

### O que é Machine Learning?
É o processo de encontrar padrões nos dados.
O modelo "aprende" quais características predizem melhor se alguém sobreviveu.

### Como funciona Random Forest?
Random Forest cria várias **árvores de decisão** e combina seus votos.

Exemplo de uma árvore simples:
```
É mulher?
├─ SIM → Provavelmente sobreviveu (probabilidade 73%)
└─ NÃO → 
    É criança?
    ├─ SIM → Provavelmente sobreviveu (probabilidade 65%)
    └─ NÃO → Provavelmente não sobreviveu (probabilidade 19%)
```

Random Forest cria 100 dessas árvores com variações aleatórias.
A resposta final é a **maioria dos votos** das árvores.

In [7]:
print("=" * 80)
print("TREINAR O MODELO FINAL")
print("=" * 80)

print("""
Agora que validamos que o ensemble funciona bem,
vamos treinar no **dataset completo** para gerar previsões mais robustas.

Usaremos todos os dados de treino disponíveis.
""")

# Treinar modelo final em todos os dados de treino
print("Treinando ensemble de modelos (Random Forest + Gradient Boosting + Extra Trees + AdaBoost + Logistic Regression)...")
model.fit(X_train, y_train)
print("✓ Modelo treinado com sucesso!")

# Calcular acurácia no próprio conjunto de treino (para referência)
train_predictions = model.predict(X_train)
train_accuracy = accuracy_score(y_train, train_predictions)

print(f"\nAcurácia no conjunto de treino: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")

print("""
Nota: A acurácia no treino é esperada ser um pouco maior do que na validação cruzada,
pois o modelo foi treinado nesses dados.
""")

# Importância das features
print("\n" + "=" * 80)
print("IMPORTÂNCIA DAS FEATURES (VARIÁVEIS)")
print("=" * 80)
print("""
O ensemble não fornece importâncias diretas,
mas podemos extrair as importâncias do Random Forest base.
""")

if hasattr(model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': model.feature_importances_
    })
    feature_importance = feature_importance.sort_values('Importance', ascending=False)
    print(feature_importance.head(10).to_string(index=False))
    print(f"\nTotal de features: {len(feature_importance)}")
elif hasattr(model, 'named_estimators_') and 'rf' in model.named_estimators_:
    rf = model.named_estimators_['rf']
    if hasattr(rf, 'feature_importances_'):
        feature_importance = pd.DataFrame({
            'Feature': X_train.columns,
            'Importance': rf.feature_importances_
        })
        feature_importance = feature_importance.sort_values('Importance', ascending=False)
        print(feature_importance.head(10).to_string(index=False))
        print(f"\nTotal de features: {len(feature_importance)}")
    else:
        print("O Random Forest base não fornece importâncias de features.")
else:
    print("Este modelo não fornece importâncias de features diretamente.")
print("""
INTERPRETAÇÃO:
- Sex é a mais importante (faz sentido: mulheres tinham prioridade)
- Pclass é segunda (classe social influenciava localização e acesso aos botes)
- Fare está alta (relacionado a classe)
- Age também é importante (crianças tinham prioridade)
""")


TREINAR O MODELO FINAL

Agora que validamos que o ensemble funciona bem,
vamos treinar no **dataset completo** para gerar previsões mais robustas.

Usaremos todos os dados de treino disponíveis.

Treinando ensemble de modelos (Random Forest + Gradient Boosting + Extra Trees + AdaBoost + Logistic Regression)...
✓ Modelo treinado com sucesso!

Acurácia no conjunto de treino: 0.8866 (88.66%)

Nota: A acurácia no treino é esperada ser um pouco maior do que na validação cruzada,
pois o modelo foi treinado nesses dados.


IMPORTÂNCIA DAS FEATURES (VARIÁVEIS)

O ensemble não fornece importâncias diretas,
mas podemos extrair as importâncias do Random Forest base.

    Feature  Importance
   Title_Mr    0.183512
        Sex    0.169962
       Fare    0.148398
        Age    0.105347
     Pclass    0.093683
 FamilySize    0.060854
 Title_Miss    0.051219
  Title_Mrs    0.044854
CabinDeck_X    0.038664
 Embarked_S    0.019471

Total de features: 24

INTERPRETAÇÃO:
- Sex é a mais importante (faz s

## Passo 7: Fazer Previsões no Conjunto de Teste

### O que fazemos?
Usamos nosso modelo treinado para **prever** quem sobreviveu no conjunto de teste.
Essas são as previsões que vamos submeter.

In [8]:
print("=" * 80)
print("FAZER PREVISÕES NO CONJUNTO DE TESTE")
print("=" * 80)

# Fazer previsões
print("\nFazendo previsões para o conjunto de teste...")
test_predictions = model.predict(X_test)
print(f"✓ Previsões feitas para {len(test_predictions)} passageiros")

# Mostrar alguns exemplos
print("\nPrimeiras 10 previsões:")
print("Índice | Previsão | Significado")
print("-" * 40)
for i in range(min(10, len(test_predictions))):
    pred = test_predictions[i]
    meaning = "Sobreviveu" if pred == 1 else "Não sobreviveu"
    print(f"{i:5} | {pred:8} | {meaning}")

# Estatísticas das previsões
print(f"\nEstatísticas das previsões:")
print(f"Previsões de sobrevivência (1): {sum(test_predictions)} pessoas ({100*sum(test_predictions)/len(test_predictions):.2f}%)")
print(f"Previsões de não-sobrevivência (0): {len(test_predictions) - sum(test_predictions)} pessoas ({100*(len(test_predictions) - sum(test_predictions))/len(test_predictions):.2f}%)")

FAZER PREVISÕES NO CONJUNTO DE TESTE

Fazendo previsões para o conjunto de teste...
✓ Previsões feitas para 418 passageiros

Primeiras 10 previsões:
Índice | Previsão | Significado
----------------------------------------
    0 |        0 | Não sobreviveu
    1 |        0 | Não sobreviveu
    2 |        0 | Não sobreviveu
    3 |        0 | Não sobreviveu
    4 |        1 | Sobreviveu
    5 |        0 | Não sobreviveu
    6 |        1 | Sobreviveu
    7 |        0 | Não sobreviveu
    8 |        1 | Sobreviveu
    9 |        0 | Não sobreviveu

Estatísticas das previsões:
Previsões de sobrevivência (1): 158 pessoas (37.80%)
Previsões de não-sobrevivência (0): 260 pessoas (62.20%)


## Passo 8: Salvar Submissão

### O que é um arquivo de submissão?
É um arquivo CSV com dois colunass:
- **PassengerId** - identificação única do passageiro
- **Survived** - nossa previsão (0 ou 1)

Este arquivo é enviado para o Kaggle para avaliação.

In [9]:
print("=" * 80)
print("SALVAR ARQUIVO DE SUBMISSÃO")
print("=" * 80)

# Criar DataFrame com PassengerId e Survived
submission = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],
    'Survived': test_predictions
})

print("\nPrimeiras linhas do arquivo de submissão:")
print(submission.head())

# Salvar para CSV
submission.to_csv('submission_melhorado8.csv', index=False)
print(f"\n✓ Arquivo salvo como 'submission_melhorado8.csv'")
print(f"  Total de linhas: {len(submission)}")
print(f"  Colunas: {list(submission.columns)}")

print("""
Este arquivo está pronto para ser enviado ao Kaggle!
Pode comparar com o resultado original em 'submission.csv'.
""")

SALVAR ARQUIVO DE SUBMISSÃO

Primeiras linhas do arquivo de submissão:
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1

✓ Arquivo salvo como 'submission_melhorado8.csv'
  Total de linhas: 418
  Colunas: ['PassengerId', 'Survived']

Este arquivo está pronto para ser enviado ao Kaggle!
Pode comparar com o resultado original em 'submission.csv'.



## Resumo e Conclusões

### O que aprendemos?

1. **Carregamento de Dados** - entender estrutura, tipos, valores faltantes
2. **Pré-processamento** - tratar dados faltantes e inconsistentes
3. **Feature Engineering** - criar novas variáveis úteis
4. **Codificação** - converter texto em números
5. **Validação Cruzada** - testar confiabilidade do modelo
6. **Modelagem** - treinar um algoritmo para aprender padrões
7. **Previsão** - usar o modelo em dados novos
8. **Submissão** - formatar e salvar resultados

### Melhorias aplicadas vs. modelo original:

| Aspecto | Original | Melhorado |
|---------|----------|----------|
| Features | 4 | 15+ |
| Tratamento de NaNs | Nenhum | Completo |
| Feature Engineering | Não | Sim |
| Validação | Não | Validação Cruzada |
| Features criadas | Nenhuma | Title, FamilySize, IsAlone, CabinDeck, AgeGroup |

### Próximos passos para melhorar ainda mais:

1. **Testar outros modelos**: GradientBoosting, XGBoost, LightGBM
2. **Ajuste de hiperparâmetros**: GridSearchCV para encontrar melhor configuração
3. **Ensemble**: Combinar previsões de múltiplos modelos
4. **Feature Selection**: Identificar e remover features irrelevantes
5. **Análise de erros**: Ver quais passageiros o modelo erra e por quê
6. **Scaling**: Normalizar variáveis contínuas (pode ajudar alguns modelos)